In [0]:
-- ============================================================
-- Food Inspection Analytics — SQL Testing Scripts
-- Run in Databricks SQL Editor or a notebook SQL cell
-- Schema: default (workspace catalog)
-- ============================================================

-- ============================================================
-- SECTION 1 — BRONZE LAYER TESTS
-- ============================================================

-- 1.1 Row counts
SELECT 'bronze_chicago' AS tbl, COUNT(*) AS rows FROM bronze_chicago
UNION ALL
SELECT 'bronze_dallas',          COUNT(*)        FROM bronze_dallas;

-- 1.2 Chicago — check for null critical fields in raw data
SELECT
    SUM(CASE WHEN dba_name        IS NULL THEN 1 ELSE 0 END) AS null_dba_name,
    SUM(CASE WHEN inspection_date IS NULL THEN 1 ELSE 0 END) AS null_inspection_date,
    SUM(CASE WHEN inspection_type IS NULL THEN 1 ELSE 0 END) AS null_inspection_type,
    SUM(CASE WHEN zip             IS NULL THEN 1 ELSE 0 END) AS null_zip,
    SUM(CASE WHEN results         IS NULL THEN 1 ELSE 0 END) AS null_results,
    SUM(CASE WHEN violations      IS NULL THEN 1 ELSE 0 END) AS null_violations
FROM bronze_chicago;

-- 1.3 Dallas — check for null critical fields in raw data
SELECT
    SUM(CASE WHEN restaurant_name    IS NULL THEN 1 ELSE 0 END) AS null_name,
    SUM(CASE WHEN inspection_date    IS NULL THEN 1 ELSE 0 END) AS null_date,
    SUM(CASE WHEN inspection_type    IS NULL THEN 1 ELSE 0 END) AS null_type,
    SUM(CASE WHEN zip_code           IS NULL THEN 1 ELSE 0 END) AS null_zip,
    SUM(CASE WHEN inspection_score   IS NULL THEN 1 ELSE 0 END) AS null_score,
    MIN(inspection_score)                                        AS min_score,
    MAX(inspection_score)                                        AS max_score
FROM bronze_dallas;

-- 1.4 Dallas — confirm score = 100 - sum(violation points) for sample
-- Expected: all rows return 0 difference
SELECT
    restaurant_name,
    inspection_score,
    (COALESCE(CAST(`violation_points_-_1` AS INT), 0) +
     COALESCE(CAST(`violation_points_-_2` AS INT), 0) +
     COALESCE(CAST(`violation_points_-_3` AS INT), 0)) AS sample_pts,
    inspection_score - (100 -
     COALESCE(CAST(`violation_points_-_1` AS INT), 0) -
     COALESCE(CAST(`violation_points_-_2` AS INT), 0) -
     COALESCE(CAST(`violation_points_-_3` AS INT), 0)) AS diff
FROM bronze_dallas
LIMIT 10;

-- 1.5 Chicago — distinct results values
SELECT results, COUNT(*) AS cnt
FROM bronze_chicago
GROUP BY results
ORDER BY cnt DESC;

-- 1.6 Dallas — violation_points_-_1 data type verification
-- Should be INT. If returns non-numeric cast errors, Bronze inferSchema issue.
SELECT
    COUNT(*)                                                          AS total_rows,
    SUM(CASE WHEN `violation_points_-_1` IS NULL THEN 1 ELSE 0 END) AS null_pts_1,
    SUM(CASE WHEN `violation_points_-_2` IS NULL THEN 1 ELSE 0 END) AS null_pts_2
FROM bronze_dallas;


-- ============================================================
-- SECTION 2 — SILVER LAYER TESTS
-- ============================================================

-- 2.1 Row counts — compare Bronze vs Silver drop rate
SELECT 'bronze_chicago' AS layer, COUNT(*) AS rows FROM bronze_chicago
UNION ALL
SELECT 'silver_chicago',          COUNT(*)        FROM silver_chicago
UNION ALL
SELECT 'bronze_dallas',           COUNT(*)        FROM bronze_dallas
UNION ALL
SELECT 'silver_dallas',           COUNT(*)        FROM silver_dallas;

-- 2.2 Chicago Silver — confirm renamed columns exist
SELECT
    COUNT(business_name)     AS has_business_name,
    COUNT(license_number)    AS has_license_number,
    COUNT(zip_code)          AS has_zip_code,
    COUNT(inspection_result) AS has_inspection_result,
    COUNT(inspection_score)  AS has_inspection_score,
    COUNT(source_city)       AS has_source_city,
    COUNT(inspection_date)   AS has_inspection_date
FROM silver_chicago;

-- 2.3 Chicago Silver — validate derived scores match result mapping
-- Expected: every result maps to exactly one score value
SELECT inspection_result, inspection_score, COUNT(*) AS cnt
FROM silver_chicago
GROUP BY inspection_result, inspection_score
ORDER BY inspection_result;
-- Pass=90, Pass w/ Conditions=80, Fail=70, No Entry=0, others=NULL

-- 2.4 Chicago Silver — no false pass rule
-- Expected: 0 rows
SELECT COUNT(*) AS false_pass_violations
FROM silver_chicago
WHERE inspection_result = 'Pass'
  AND (violations LIKE '%PRIORITY VIOLATION%' OR violations LIKE '%CRITICAL%');

-- 2.5 Chicago Silver — zip code format
-- Expected: 0 rows
SELECT COUNT(*) AS invalid_zips
FROM silver_chicago
WHERE zip_code NOT RLIKE '^[0-9]{5}$';

-- 2.6 Chicago Silver — no future inspection dates
-- Expected: 0 rows
SELECT COUNT(*) AS future_dates
FROM silver_chicago
WHERE inspection_date > CURRENT_DATE();

-- 2.7 Dallas Silver — score range after DQX
-- Expected: min >= 0, max <= 100
SELECT
    MIN(inspection_score) AS min_score,
    MAX(inspection_score) AS max_score,
    SUM(CASE WHEN inspection_score < 0   THEN 1 ELSE 0 END) AS negative_scores,
    SUM(CASE WHEN inspection_score > 100 THEN 1 ELSE 0 END) AS over_100_scores
FROM silver_dallas;

-- 2.8 Dallas Silver — inspection_result derived correctly
-- Expected: only Pass, Pass w/ Conditions, Fail
SELECT inspection_result, COUNT(*) AS cnt
FROM silver_dallas
GROUP BY inspection_result;

-- 2.9 Dallas Silver — inspection_id exists and is unique
-- Expected: 0 duplicates
SELECT COUNT(*) - COUNT(DISTINCT inspection_id) AS duplicate_inspection_ids
FROM silver_dallas;

-- 2.10 Dallas Silver — lat/lon parsed from string
-- Expected: no nulls on rows that had coordinates in source
SELECT
    SUM(CASE WHEN latitude  IS NULL THEN 1 ELSE 0 END) AS null_lat,
    SUM(CASE WHEN longitude IS NULL THEN 1 ELSE 0 END) AS null_lon
FROM silver_dallas;


-- ============================================================
-- SECTION 3 — GOLD DIMENSION TESTS
-- ============================================================

-- 3.1 Row counts for all Gold tables
SELECT 'dim_date'            AS tbl, COUNT(*) AS rows FROM dim_date
UNION ALL SELECT 'dim_inspection_type',  COUNT(*) FROM dim_inspection_type
UNION ALL SELECT 'dim_risk',             COUNT(*) FROM dim_risk
UNION ALL SELECT 'dim_location',         COUNT(*) FROM dim_location
UNION ALL SELECT 'dim_restaurant',       COUNT(*) FROM dim_restaurant
UNION ALL SELECT 'dim_violation',        COUNT(*) FROM dim_violation
UNION ALL SELECT 'fact_inspection',      COUNT(*) FROM fact_inspection
UNION ALL SELECT 'fact_inspection_violation', COUNT(*) FROM fact_inspection_violation;

-- 3.2 dim_date — no gaps in date spine
-- Expected: max_date - min_date + 1 = total rows
SELECT
    MIN(full_date)                                      AS min_date,
    MAX(full_date)                                      AS max_date,
    COUNT(*)                                            AS total_rows,
    DATEDIFF(MAX(full_date), MIN(full_date)) + 1        AS expected_rows,
    COUNT(*) - (DATEDIFF(MAX(full_date), MIN(full_date)) + 1) AS gap_count
FROM dim_date;

-- 3.3 dim_date — PK uniqueness
-- Expected: 0
SELECT COUNT(*) - COUNT(DISTINCT date_sk) AS duplicate_date_sk FROM dim_date;

-- 3.4 dim_risk — exactly 3 rows
SELECT risk_sk, risk_code, risk_label FROM dim_risk ORDER BY risk_code;

-- 3.5 dim_restaurant — SCD2 integrity
-- No restaurant_nk should have more than one is_current = TRUE record
SELECT restaurant_nk, COUNT(*) AS current_versions
FROM dim_restaurant
WHERE is_current = TRUE
GROUP BY restaurant_nk
HAVING COUNT(*) > 1;
-- Expected: 0 rows

-- 3.6 dim_restaurant — SCD2 date continuity
-- Closed records must have end_date set; current records must have end_date NULL
SELECT
    SUM(CASE WHEN is_current = TRUE  AND end_date IS NOT NULL THEN 1 ELSE 0 END) AS current_with_end_date,
    SUM(CASE WHEN is_current = FALSE AND end_date IS NULL     THEN 1 ELSE 0 END) AS closed_without_end_date
FROM dim_restaurant;
-- Expected: both = 0

-- 3.7 dim_restaurant — split by city
SELECT source_city, COUNT(*) AS total, SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_records
FROM dim_restaurant
GROUP BY source_city;

-- 3.8 dim_violation — PK uniqueness
-- Expected: 0
SELECT COUNT(*) - COUNT(DISTINCT violation_sk) AS duplicate_violation_sk FROM dim_violation;

-- 3.9 dim_violation — grain check (code + description + city must be unique)
-- Expected: 0 rows
SELECT violation_code, violation_description, source_city, COUNT(*) AS cnt
FROM dim_violation
GROUP BY violation_code, violation_description, source_city
HAVING COUNT(*) > 1;

-- 3.10 dim_violation — codes are zero-padded (no single-digit codes)
-- Expected: 0 rows
SELECT violation_code, source_city
FROM dim_violation
WHERE LENGTH(violation_code) < 2;

-- 3.11 dim_location — grain uniqueness
-- Expected: 0 rows
SELECT street_address, zip_code, city, COUNT(*) AS cnt
FROM dim_location
GROUP BY street_address, zip_code, city
HAVING COUNT(*) > 1;

-- 3.12 dim_inspection_type — both cities present
SELECT source_city, COUNT(*) AS type_count FROM dim_inspection_type GROUP BY source_city;


-- ============================================================
-- SECTION 4 — GOLD FACT FK INTEGRITY TESTS
-- ============================================================

-- 4.1 fact_inspection — all FK null counts
-- null_restaurant_sk, null_location_sk, null_date_sk, null_type_sk = 0
-- null_risk_sk = Dallas row count (expected)
SELECT
    SUM(CASE WHEN restaurant_sk      IS NULL THEN 1 ELSE 0 END) AS null_restaurant_sk,
    SUM(CASE WHEN location_sk        IS NULL THEN 1 ELSE 0 END) AS null_location_sk,
    SUM(CASE WHEN date_sk            IS NULL THEN 1 ELSE 0 END) AS null_date_sk,
    SUM(CASE WHEN inspection_type_sk IS NULL THEN 1 ELSE 0 END) AS null_type_sk,
    SUM(CASE WHEN risk_sk            IS NULL THEN 1 ELSE 0 END) AS null_risk_sk
FROM fact_inspection;

-- 4.2 fact_inspection — orphan restaurant_sk check
-- Expected: 0 rows
SELECT COUNT(*) AS orphan_restaurant_sk
FROM fact_inspection f
LEFT JOIN dim_restaurant d ON f.restaurant_sk = d.restaurant_sk
WHERE d.restaurant_sk IS NULL;

-- 4.3 fact_inspection — orphan location_sk check
-- Expected: 0 rows
SELECT COUNT(*) AS orphan_location_sk
FROM fact_inspection f
LEFT JOIN dim_location d ON f.location_sk = d.location_sk
WHERE d.location_sk IS NULL;

-- 4.4 fact_inspection — orphan date_sk check
-- Expected: 0 rows
SELECT COUNT(*) AS orphan_date_sk
FROM fact_inspection f
LEFT JOIN dim_date d ON f.date_sk = d.date_sk
WHERE d.date_sk IS NULL;

-- 4.5 fact_inspection — null risk_sk must only be Dallas rows
-- Expected: 0 rows
SELECT COUNT(*) AS chicago_rows_with_null_risk
FROM fact_inspection
WHERE risk_sk IS NULL AND source_city = 'Chicago';

-- 4.6 fact_inspection_violation — FK null counts
-- Expected: both = 0
SELECT
    SUM(CASE WHEN inspection_sk IS NULL THEN 1 ELSE 0 END) AS null_inspection_sk,
    SUM(CASE WHEN violation_sk  IS NULL THEN 1 ELSE 0 END) AS null_violation_sk
FROM fact_inspection_violation;

-- 4.7 fact_inspection_violation — orphan inspection_sk check
-- Expected: 0 rows
SELECT COUNT(*) AS orphan_inspection_sk
FROM fact_inspection_violation fv
LEFT JOIN fact_inspection fi ON fv.inspection_sk = fi.inspection_sk
WHERE fi.inspection_sk IS NULL;

-- 4.8 fact_inspection_violation — orphan violation_sk check
-- Expected: 0 rows
SELECT COUNT(*) AS orphan_violation_sk
FROM fact_inspection_violation fv
LEFT JOIN dim_violation dv ON fv.violation_sk = dv.violation_sk
WHERE dv.violation_sk IS NULL;

-- 4.9 PK uniqueness across all fact/dim tables
SELECT 'fact_inspection'           AS tbl, COUNT(*) - COUNT(DISTINCT inspection_sk)      AS pk_dupes FROM fact_inspection
UNION ALL
SELECT 'fact_inspection_violation',        COUNT(*) - COUNT(DISTINCT insp_violation_sk)   FROM fact_inspection_violation
UNION ALL
SELECT 'dim_restaurant',                   COUNT(*) - COUNT(DISTINCT restaurant_sk)        FROM dim_restaurant
UNION ALL
SELECT 'dim_location',                     COUNT(*) - COUNT(DISTINCT location_sk)          FROM dim_location
UNION ALL
SELECT 'dim_date',                         COUNT(*) - COUNT(DISTINCT date_sk)              FROM dim_date
UNION ALL
SELECT 'dim_violation',                    COUNT(*) - COUNT(DISTINCT violation_sk)         FROM dim_violation;
-- Expected: all 0


-- ============================================================
-- SECTION 5 — BUSINESS RULE TESTS
-- ============================================================

-- 5.1 Score range by city
-- Chicago: 0 / 70 / 80 / 90 only. Dallas: 0–100.
SELECT source_city, inspection_score, COUNT(*) AS cnt
FROM fact_inspection
GROUP BY source_city, inspection_score
ORDER BY source_city, inspection_score;

-- 5.2 Chicago score-to-result mapping is consistent
-- Expected: Pass=90, Pass w/ Conditions=80, Fail=70, No Entry=0 only
SELECT inspection_result, inspection_score, COUNT(*) AS cnt
FROM fact_inspection
WHERE source_city = 'Chicago'
GROUP BY inspection_result, inspection_score
ORDER BY inspection_score DESC;

-- 5.3 Dallas score-to-result derivation is consistent
-- Expected: score>=90 always Pass, 80-89 always Pass w/ Conditions, <80 always Fail
SELECT
    inspection_result,
    MIN(inspection_score) AS min_score,
    MAX(inspection_score) AS max_score,
    COUNT(*) AS cnt
FROM fact_inspection
WHERE source_city = 'Dallas'
GROUP BY inspection_result
ORDER BY min_score DESC;

-- 5.4 Violation count per inspection (avg and distribution)
SELECT
    source_city,
    COUNT(DISTINCT fi.inspection_sk)                             AS inspections,
    COUNT(fv.insp_violation_sk)                                  AS total_violations,
    ROUND(COUNT(fv.insp_violation_sk) / COUNT(DISTINCT fi.inspection_sk), 2) AS avg_violations_per_inspection
FROM fact_inspection fi
LEFT JOIN fact_inspection_violation fv ON fi.inspection_sk = fv.inspection_sk
GROUP BY source_city;
-- Expected avg ~4.35 across both cities

-- 5.5 Dallas high score viol limit — score >= 90 must have <= 3 violations
-- Expected: 0 rows
SELECT fi.inspection_id, fi.inspection_score, COUNT(*) AS violation_count
FROM fact_inspection fi
JOIN fact_inspection_violation fv ON fi.inspection_sk = fv.inspection_sk
WHERE fi.source_city = 'Dallas'
  AND fi.inspection_score >= 90
GROUP BY fi.inspection_id, fi.inspection_score
HAVING COUNT(*) > 3;

-- 5.6 Chicago no false pass — Pass result must not have critical violations
-- Expected: 0 rows
SELECT fi.inspection_id, fi.inspection_result, dv.violation_description
FROM fact_inspection fi
JOIN fact_inspection_violation fv ON fi.inspection_sk = fv.inspection_sk
JOIN dim_violation dv ON fv.violation_sk = dv.violation_sk
WHERE fi.source_city = 'Chicago'
  AND fi.inspection_result = 'Pass'
  AND (UPPER(dv.violation_description) LIKE '%PRIORITY%'
   OR  UPPER(dv.violation_description) LIKE '%CRITICAL%');

-- 5.7 Chicago deduplication — no inspection should have the same violation code twice
-- Expected: 0 rows
SELECT fi.inspection_id, dv.violation_code, COUNT(*) AS cnt
FROM fact_inspection fi
JOIN fact_inspection_violation fv ON fi.inspection_sk = fv.inspection_sk
JOIN dim_violation dv ON fv.violation_sk = dv.violation_sk
WHERE fi.source_city = 'Chicago'
GROUP BY fi.inspection_id, dv.violation_code
HAVING COUNT(*) > 1;

-- 5.8 Dallas violation_points range — must be 1, 2, or 3 only
-- Expected: 0 rows
SELECT DISTINCT violation_points
FROM fact_inspection_violation
WHERE source_city = 'Dallas'
  AND violation_points NOT IN (1, 2, 3);

-- 5.9 Chicago violation_points must all be NULL (no point system)
-- Expected: 0 rows
SELECT COUNT(*) AS chicago_rows_with_points
FROM fact_inspection_violation
WHERE source_city = 'Chicago'
  AND violation_points IS NOT NULL;


-- ============================================================
-- SECTION 6 — DASHBOARD READINESS TESTS
-- ============================================================

-- 6.1 Pass rate by city — confirms result field is populated for both cities
SELECT
    source_city,
    COUNT(*) AS total,
    SUM(CASE WHEN inspection_result = 'Pass'             THEN 1 ELSE 0 END) AS pass_count,
    SUM(CASE WHEN inspection_result = 'Fail'             THEN 1 ELSE 0 END) AS fail_count,
    SUM(CASE WHEN inspection_result = 'Pass w/ Conditions' THEN 1 ELSE 0 END) AS conditional_count,
    ROUND(100.0 * SUM(CASE WHEN inspection_result = 'Pass' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pass_pct
FROM fact_inspection
GROUP BY source_city;

-- 6.2 Inspections per year — trend line data
SELECT
    d.year,
    fi.source_city,
    COUNT(*) AS inspections
FROM fact_inspection fi
JOIN dim_date d ON fi.date_sk = d.date_sk
GROUP BY d.year, fi.source_city
ORDER BY d.year, fi.source_city;

-- 6.3 Top 10 violations by frequency
SELECT
    dv.violation_code,
    dv.violation_description,
    dv.source_city,
    COUNT(*) AS occurrences
FROM fact_inspection_violation fv
JOIN dim_violation dv ON fv.violation_sk = dv.violation_sk
GROUP BY dv.violation_code, dv.violation_description, dv.source_city
ORDER BY occurrences DESC
LIMIT 10;

-- 6.4 Risk category breakdown — Chicago only
SELECT
    dr.risk_label,
    COUNT(*) AS inspections,
    SUM(CASE WHEN fi.inspection_result = 'Pass' THEN 1 ELSE 0 END) AS pass_count,
    ROUND(100.0 * SUM(CASE WHEN fi.inspection_result = 'Pass' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pass_pct
FROM fact_inspection fi
JOIN dim_risk dr ON fi.risk_sk = dr.risk_sk
WHERE fi.source_city = 'Chicago'
GROUP BY dr.risk_label
ORDER BY dr.risk_code;

-- 6.5 Inspection report — sample drill-through for one business
-- Replace the business_name value with any restaurant from your data
SELECT
    fi.inspection_id,
    d.full_date             AS inspection_date,
    dit.inspection_type_name,
    fi.inspection_result,
    fi.inspection_score,
    fi.license_number,
    dr.business_name,
    dr.facility_type,
    dl.street_address,
    dl.city,
    dl.zip_code
FROM fact_inspection fi
JOIN dim_restaurant     dr  ON fi.restaurant_sk     = dr.restaurant_sk AND dr.is_current = TRUE
JOIN dim_date           d   ON fi.date_sk            = d.date_sk
JOIN dim_inspection_type dit ON fi.inspection_type_sk = dit.inspection_type_sk
JOIN dim_location       dl  ON fi.location_sk        = dl.location_sk
WHERE dr.business_name LIKE '%DAIRY QUEEN%'
ORDER BY d.full_date DESC
LIMIT 10;

-- 6.6 Violation detail — for one inspection (replace inspection_id)
SELECT
    dv.violation_code,
    dv.violation_description,
    fv.violation_points,
    fv.inspector_comment
FROM fact_inspection fi
JOIN fact_inspection_violation fv ON fi.inspection_sk = fv.inspection_sk
JOIN dim_violation dv ON fv.violation_sk = dv.violation_sk
WHERE fi.inspection_id = '2633339'   -- replace with any inspection_id
ORDER BY dv.violation_code;

-- 6.7 Map data availability — rows with valid coordinates
SELECT
    fi.source_city,
    COUNT(*)                                                        AS total_inspections,
    SUM(CASE WHEN dl.latitude IS NOT NULL THEN 1 ELSE 0 END)       AS with_coordinates,
    SUM(CASE WHEN dl.latitude IS NULL     THEN 1 ELSE 0 END)       AS without_coordinates,
    ROUND(100.0 * SUM(CASE WHEN dl.latitude IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_mappable
FROM fact_inspection fi
JOIN dim_location dl ON fi.location_sk = dl.location_sk
GROUP BY fi.source_city;

-- 6.8 Average score by inspection type — confirms inspection_type_sk joins correctly
SELECT
    dit.inspection_type_name,
    dit.source_city,
    COUNT(*)                        AS inspections,
    ROUND(AVG(fi.inspection_score), 1) AS avg_score
FROM fact_inspection fi
JOIN dim_inspection_type dit ON fi.inspection_type_sk = dit.inspection_type_sk
GROUP BY dit.inspection_type_name, dit.source_city
ORDER BY dit.source_city, avg_score DESC;